<a href="https://colab.research.google.com/github/AlejandroVargasAraujo/econ-p/blob/main/MONETARIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Analisis Teórico Preeliminar al VAR Generalizado
---

## Cambio estructural y Estacionariedad
###Prueba Zivot-Andrews

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.formula.api import ols
import warnings
warnings.filterwarnings('ignore')

# ── Cargar datos ──────────────────────────────────────────────────────────────
NP = pd.read_excel('/content/Datos_ProyectoMonetaria.xlsx')
NP['Fecha'] = pd.to_datetime(NP['Fecha'])
NP = NP.set_index('Fecha')

series_list = ['lvol_ytm', 'log_vol_fx_oil', 'log_vol_fx_tc']
nombres = {
    'lvol_ytm':       'Log-Vol Bono Soberano (YTM)',
    'log_vol_fx_oil': 'Log-Vol Precio Petróleo',
    'log_vol_fx_tc':  'Log-Vol Tipo de Cambio',
}

# ── Función Zivot-Andrews (modelo A: quiebre en intercepto) ───────────────────
def ZivotAndrewsA(serie, k=8):
    dta = NP[[serie]].dropna().copy()
    dta.rename(columns={serie: 'y'}, inplace=True)
    dta['t']  = np.arange(dta.shape[0])
    dta['Ly'] = dta['y'].shift(1)
    dta['Dy'] = dta['y'].diff(1)
    for j in range(1, k + 1):
        dta[f'D{j}y'] = dta['Dy'].shift(j)

    lags = '+'.join(dta.columns[-k:])

    # Búsqueda del punto de quiebre óptimo
    # Excluir el 15% inicial y final (trim estándar Zivot-Andrews)
    n = dta.shape[0]
    trim = int(np.ceil(0.15 * n))
    avals = pd.Series(0.0, index=dta.index[trim:-trim])

    for tau in avals.index:
        dta['DL'] = (dta.index > tau).astype(int)
        reg = ols('y ~ Ly + t + DL + ' + lags, dta).fit()
        avals[tau] = ((reg.params - 1) / reg.bse)['Ly']

    tauhat = avals.idxmin()
    tval   = avals.min()

    dta['DL'] = (dta.index > tauhat).astype(int)
    reg = ols('y ~ Ly + t + DL + ' + lags, dta).fit()

    return {
        r'$\\hat{T}_B$': tauhat,
        r'$\\alpha_1$':  reg.params['Ly'],
        r'$t$':         tval,
    }

# ── Valores críticos Zivot-Andrews (Modelo A, tabla original) ─────────────────
vc = {1: -5.34, 5: -4.80, 10: -4.58}

# ── Ejecutar y mostrar resultados ─────────────────────────────────────────────
print("=" * 70)
print("PRUEBA DE ZIVOT-ANDREWS (Modelo A – Quiebre en Intercepto)")
print("=" * 70)

resultados = {}
for s in series_list:
    res = ZivotAndrewsA(s)
    resultados[s] = res
    nombre = nombres[s]
    tquiebre = res[r'$\\hat{T}_B$']
    alpha1   = res[r'$\\alpha_1$']
    tstat    = res[r'$t$']

    # Decisión al 5%
    rechazo = tstat < vc[5]
    decision = "RECHAZA H₀ (hay quiebre + estacionariedad)" if rechazo \
               else "NO rechaza H₀ (raíz unitaria)"

    print(f"\nSerie: {nombre}")
    print(f"  Fecha de quiebre estimada : {tquiebre.date()}")
    print(f"  α₁ (coef. Ly)             : {alpha1:.6f}")
    print(f"  t-estadístico (Zivot-A)   : {tstat:.4f}")
    print(f"  Valores críticos (1%/5%/10%): {vc[1]} / {vc[5]} / {vc[10]}")
    print(f"  Decisión (5%)             : {decision}")

print("\n" + "=" * 70)
print("Nota: H₀ = raíz unitaria con quiebre estructural (Zivot & Andrews 1992)")
print("      El rechazo implica estacionariedad alrededor de una tendencia")
print("      con un quiebre estructural en la fecha estimada.")
print("=" * 70)

# ── Resumen en DataFrame ───────────────────────────────────────────────────────
rows = []
for s in series_list:
    r = resultados[s]
    rows.append({
        'Serie': nombres[s],
        'Fecha quiebre': r[r'$\\hat{T}_B$'].date(),
        'α₁':            round(r[r'$\\alpha_1$'], 6),
        't-stat (ZA)':   round(r[r'$t$'], 4),
        'VC 1%':  vc[1],
        'VC 5%':  vc[5],
        'VC 10%': vc[10],
        'Decisión 5%': 'Rechaza H₀' if r[r'$t$'] < vc[5] else 'No rechaza H₀',
    })

resumen = pd.DataFrame(rows)
print("\nRESUMEN:\n")
print(resumen.to_string(index=False))

PRUEBA DE ZIVOT-ANDREWS (Modelo A – Quiebre en Intercepto)

Serie: Log-Vol Bono Soberano (YTM)
  Fecha de quiebre estimada : 2023-10-13
  α₁ (coef. Ly)             : 0.229441
  t-estadístico (Zivot-A)   : -8.1908
  Valores críticos (1%/5%/10%): -5.34 / -4.8 / -4.58
  Decisión (5%)             : RECHAZA H₀ (hay quiebre + estacionariedad)

Serie: Log-Vol Precio Petróleo
  Fecha de quiebre estimada : 2024-02-02
  α₁ (coef. Ly)             : 0.667198
  t-estadístico (Zivot-A)   : -5.9090
  Valores críticos (1%/5%/10%): -5.34 / -4.8 / -4.58
  Decisión (5%)             : RECHAZA H₀ (hay quiebre + estacionariedad)

Serie: Log-Vol Tipo de Cambio
  Fecha de quiebre estimada : 2023-10-12
  α₁ (coef. Ly)             : 0.575099
  t-estadístico (Zivot-A)   : -6.0996
  Valores críticos (1%/5%/10%): -5.34 / -4.8 / -4.58
  Decisión (5%)             : RECHAZA H₀ (hay quiebre + estacionariedad)

Nota: H₀ = raíz unitaria con quiebre estructural (Zivot & Andrews 1992)
      El rechazo implica estacionarie

---

##Selección de Rezagos /Estimación del VAR mediante OLS / Estabilidad (Eigenvectores)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from statsmodels.tsa.api import VAR
import warnings
import os # Import the os module
warnings.filterwarnings('ignore')

# ── Cargar datos ──────────────────────────────────────────────────────────────
NP = pd.read_excel('/content/Datos_ProyectoMonetaria.xlsx')
NP['Fecha'] = pd.to_datetime(NP['Fecha'])
NP = NP.set_index('Fecha')

nombres = {
    'lvol_ytm':       'Log-Vol Bono Soberano',
    'log_vol_fx_oil': 'Log-Vol Precio Petróleo',
    'log_vol_fx_tc':  'Log-Vol Tipo de Cambio',
}

# Create the output directory if it doesn't exist
output_dir = '/content/outputs/'
os.makedirs(output_dir, exist_ok=True)

# ── 1. Gráfica de las series ──────────────────────────────────────────────────
fig, axs = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, col in zip(axs, NP.columns):
    ax.plot(NP.index, NP[col], linewidth=0.9, color='steelblue')
    ax.set_title(nombres[col], fontsize=11)
    ax.set_ylabel('Log-Volatilidad')
    ax.grid(alpha=0.3)
fig.suptitle('Series de Log-Volatilidad Diaria', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'series.png'), dpi=150, bbox_inches='tight')
plt.close()

# ── 2. Selección de rezagos ───────────────────────────────────────────────────
model = VAR(NP)
seleccion = model.select_order(10)
print("=" * 60)
print("SELECCIÓN DE REZAGOS DEL VAR")
print("=" * 60)
print(seleccion.summary())

# Rezago óptimo por criterio
p_aic  = seleccion.aic
p_bic  = seleccion.bic
p_hqic = seleccion.hqic
print(f"\nRezago óptimo por criterio:")
print(f"  AIC  → p = {p_aic}")
print(f"  BIC  → p = {p_bic}")
print(f"  HQIC → p = {p_hqic}")
print(f"\n→ Se usará p = {p_aic} (AIC) para la estimación.")

# ── 3. Estimación del VAR(p) ──────────────────────────────────────────────────
p = p_aic
fitted = model.fit(p)

# ── 4. Matriz Omega (Sigma_u) ─────────────────────────────────────────────────
#
# ¿Qué es Omega (Σ_u)?
# Es la matriz de varianza-covarianza de los RESIDUOS del VAR estimado.
# Cada elemento (i,j) mide la covarianza entre el error de la ecuación i
# y el error de la ecuación j en el mismo período t.
# La diagonal son las varianzas de cada ecuación por separado.
# NO es la descomposición de la varianza del pronóstico (FEVD);
# esa es otra cosa que se calcula después con IRF.
# Omega se usa para:
#   a) Descomposición de Cholesky → ortogonalizar shocks para IRF
#   b) Estimar la correlación contemporánea entre variables
#
Omega = fitted.sigma_u  # ya es un DataFrame con nombres originales de columnas
# Renombramos filas y columnas para mostrar nombres legibles
Omega_df = Omega.rename(index=nombres, columns=nombres)

print("\n" + "=" * 60)
print("MATRIZ OMEGA — Varianza-Covarianza de Residuos (Σ_u)")
print("=" * 60)
print(Omega_df.round(6))

# Descomposición de Cholesky de Omega: Omega = P @ P.T
# Operamos sobre el array numpy subyacente para evitar problemas de índices
Omega_arr = Omega.values
P_chol = np.linalg.cholesky(Omega_arr)
P_df = pd.DataFrame(P_chol, index=list(nombres.values()), columns=list(nombres.values()))
print("\nDescomposición de Cholesky (P triangular inferior, Omega = P·Pᵀ):")
print(P_df.round(6))
print("\nVerificación — P·Pᵀ ≈ Omega:")
print(pd.DataFrame(P_chol @ P_chol.T,
                   index=list(nombres.values()),
                   columns=list(nombres.values())).round(6))

# ── 5. Eigenvalores y estabilidad ─────────────────────────────────────────────
#
# El VAR(p) es ESTABLE si todos los eigenvalores de la matriz compañera
# tienen módulo ESTRICTAMENTE menor que 1 → caen dentro del círculo unitario.
# statsmodels reporta las RAÍCES del polinomio característico (inversas de
# los eigenvalores). La condición de estabilidad equivale a que esas raíces
# tengan módulo > 1 (fuera del círculo unitario en ese espacio).
# Nosotros graficamos los eigenvalores (1/roots) dentro del círculo.
#
roots = fitted.roots
eigenvalues = 1.0 / roots

print("\n" + "=" * 60)
print(f"ESTABILIDAD DEL VAR({p})")
print("=" * 60)
print(f"{'i':<5} {'Eigenvalor (Re)':<20} {'Eigenvalor (Im)':<20} {'|módulo|':<12}")
print("-" * 57)
for i, ev in enumerate(eigenvalues):
    print(f"{i+1:<5} {ev.real:<20.6f} {ev.imag:<20.6f} {abs(ev):<12.6f}")

estable = fitted.is_stable()
print(f"\n¿VAR({p}) es estable? → {'SÍ ✓ (todos |λ| < 1)' if estable else 'NO ✗ (algún |λ| ≥ 1)'}")

# ── 6. Gráfico del círculo unitario ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))

# Círculo unitario
theta = np.linspace(0, 2 * np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=1.2, label='Círculo unitario')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')

# Eigenvalores
reales = eigenvalues[eigenvalues.imag == 0]
complejos = eigenvalues[eigenvalues.imag != 0]

ax.scatter(reales.real, reales.imag,
           color='steelblue', zorder=5, s=70, label='Eigenvalor real')
ax.scatter(complejos.real, complejos.imag,
           color='tomato', zorder=5, s=70, marker='^', label='Eigenvalor complejo')

ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect('equal')
ax.set_xlabel('Parte Real')
ax.set_ylabel('Parte Imaginaria')
ax.set_title(f'Eigenvalores del VAR({p}) — Círculo Unitario\n'
             f'{"✓ VAR Estable: todos los eigenvalores dentro del círculo" if estable else "✗ VAR Inestable"}',
             fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'circulo_unitario.png'), dpi=150, bbox_inches='tight')
plt.close()

print("\nGráficos guardados:")
print("  → series.png")
print("  → circulo_unitario.png")

SELECCIÓN DE REZAGOS DEL VAR
 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -2.626      -2.606     0.07236      -2.618
1       -2.928      -2.847     0.05352      -2.897
2       -3.060     -2.919*     0.04691      -3.005
3       -3.104      -2.903     0.04486     -3.026*
4      -3.121*      -2.859    0.04414*      -3.019
5       -3.120      -2.798     0.04418      -2.995
6       -3.105      -2.724     0.04481      -2.958
7       -3.097      -2.655     0.04519      -2.926
8       -3.091      -2.589     0.04545      -2.897
9       -3.078      -2.516     0.04605      -2.860
10      -3.071      -2.448     0.04640      -2.829
--------------------------------------------------

Rezago óptimo por criterio:
  AIC  → p = 4
  BIC  → p = 2
  HQIC → p = 3

→ Se usará p = 4 (AIC) para la estimación.

MATRIZ OMEGA — Varianza-Covarianza de Residuos (Σ_u)
                         Log-Vol B

---

##Prueba de Causalidad de Granger

Granger ≠ Conectividad financiera



Granger mide predictibilidad lineal con rezagos: ¿el pasado de X ayuda a predecir el futuro de Y? Si no hay causalidad de Granger, solo significa que no hay transmisión dinámica rezagada entre variables.
Pero conectividad financiera es más amplia que eso. Puede existir perfectamente aunque Granger no encuentre nada.

In [ ]:

# ── 7. Causalidad de Granger ──────────────────────────────────────────────────
#
# La prueba de Granger NO usa Cholesky. Es una prueba F estándar sobre los
# coeficientes del VAR: verifica si los rezagos de X ayudan a predecir Y
# más allá de lo que Y se predice a sí misma.
# H₀: X NO causa en el sentido de Granger a Y
# Si p < 0.05 → rechazamos H₀ → X SÍ tiene poder predictivo sobre Y
#
vars_orig   = ['lvol_ytm', 'log_vol_fx_oil', 'log_vol_fx_tc']
vars_nombre = {
    'lvol_ytm':       'Bono Soberano',
    'log_vol_fx_oil': 'Precio Petróleo',
    'log_vol_fx_tc':  'Tipo de Cambio',
}

print("\n" + "=" * 60)
print(f"CAUSALIDAD DE GRANGER — VAR({p})")
print("H₀: X no causa (en Granger) a Y")
print("=" * 60)

pares = [(x, y) for x in vars_orig for y in vars_orig if x != y]

filas = []
for causing, caused in pares:
    res = fitted.test_causality(caused, causing)
    filas.append({
        'X (causa)':   vars_nombre[causing],
        'Y (efecto)':  vars_nombre[caused],
        'F-stat':      round(res.test_statistic, 4),
        'p-valor':     round(res.pvalue, 4),
        'Decisión 5%': 'Causa ✓' if res.pvalue < 0.05 else 'No causa ✗',
    })

resumen_granger = pd.DataFrame(filas)
print(resumen_granger.to_string(index=False))

print("\nDetalle por par:")
for causing, caused in pares:
    print(f"\n  {vars_nombre[causing]} → {vars_nombre[caused]}")
    print(fitted.test_causality(caused, causing).summary())



CAUSALIDAD DE GRANGER — VAR(4)
H₀: X no causa (en Granger) a Y
      X (causa)      Y (efecto)  F-stat  p-valor Decisión 5%
  Bono Soberano Precio Petróleo  1.9478   0.1000  No causa ✗
  Bono Soberano  Tipo de Cambio  0.9956   0.4086  No causa ✗
Precio Petróleo   Bono Soberano  1.2140   0.3028  No causa ✗
Precio Petróleo  Tipo de Cambio  0.1200   0.9754  No causa ✗
 Tipo de Cambio   Bono Soberano  1.1720   0.3212  No causa ✗
 Tipo de Cambio Precio Petróleo  1.6228   0.1658  No causa ✗

Detalle por par:

  Bono Soberano → Precio Petróleo
Granger causality F-test. H_0: lvol_ytm does not Granger-cause log_vol_fx_oil. Conclusion: fail to reject H_0 at 5% significance level.
Test statistic Critical value p-value          df        
---------------------------------------------------------
         1.948          2.376   0.100 (4, np.int64(2001))
---------------------------------------------------------

  Bono Soberano → Tipo de Cambio
Granger causality F-test. H_0: lvol_ytm does not Grang